# KNative Messaging Library Demo

This notebook demonstrates the `messaging` library — a transport-agnostic
Python library for publishing and handling CloudEvents via a simple `MessageBus` API.

The bus runs inside the cluster alongside the KNative Broker, so events
published here flow through the real infrastructure:

```
Notebook → MessageBus → KNative Broker → Triggers → Subscribers
```

## 1. Setup — Create a MessageBus with KNative Transport

In [ ]:
import os
from messaging import MessageBus, CloudEvent, Disposition, MessageContext
from messaging.knative import KNativeEventingPublisher

BROKER_URL = os.environ.get(
    "BROKER_URL",
    "http://kafka-broker-ingress.knative-eventing.svc.cluster.local/default/default"
)

bus = MessageBus()
bus.configure(KNativeEventingPublisher(broker_url=BROKER_URL))

print(f"✅ MessageBus configured with KNative transport")
print(f"   Broker URL: {BROKER_URL}")

## 2. Publish a CloudEvent

Events are published as CloudEvents (CE spec v1.0). The library handles
the HTTP transport and CE envelope automatically.

The event will appear in the **Interactive Demo** tab (left panel) since
the Trigger routes `com.example.demo` events to the demo backend.

In [ ]:
event = CloudEvent(
    type="com.example.demo",
    source="/demo/jupyter",
    data={"message": "Hello from Jupyter!", "origin": "notebook"}
)

await bus.publish(event)
print(f"📤 Published event: {event.id}")
print(f"   Type: {event.type}")
print(f"   Source: {event.source}")
print(f"   → Check the Interactive Demo tab!")

## 3. Publish to Azure Service Bus (via Broker)

To route an event to ASB, publish with type `asb.outbound`.
The `broker-to-asb` Camel-K integration picks it up and forwards
it to the `knative-outbound` queue.

Watch the **ASB Explorer** panel on the right!

In [ ]:
asb_event = CloudEvent(
    type="asb.outbound",
    source="/demo/jupyter",
    data={"message": "Hello ASB from Jupyter!", "destination": "knative-outbound"}
)

await bus.publish(asb_event)
print(f"📤 Published to ASB via Broker: {asb_event.id}")
print(f"   → Check ASB Explorer → knative-outbound queue")

## 4. Live Event Stream

Start a live event stream that displays CloudEvents as they arrive
at the demo backend. This connects via WebSocket and renders events
in real-time — try publishing from another cell or from the Interactive Demo tab!

In [ ]:
from event_stream import EventStream

stream = EventStream()
stream.start()

# Events will appear below as they arrive.
# Publish from another cell, or use the Interactive Demo tab!
# Call stream.stop() when done.

## 5. Mount as FastAPI Router

In a real application, the bus exposes an HTTP endpoint that receives
CloudEvents from KNative Triggers:

```python
from fastapi import FastAPI
from messaging import MessageBus
from messaging.knative import KNativeEventingPublisher

app = FastAPI()
bus = MessageBus()

# Register handlers
@bus.handler("order.created")
async def on_order(event, ctx):
    # process the order...
    return Disposition.COMPLETE

# Mount the bus router — receives events from Triggers
app.include_router(bus.router, prefix="/events")

# Configure transport for publishing
bus.configure(KNativeEventingPublisher(broker_url="http://..."))
```

The library is **transport-agnostic** — the same handler code works whether
events flow through KNative, Kafka, or any CloudEvents-compatible broker.

## 6. Cleanup

In [ ]:
await bus.close()
print("🔒 Bus transport closed")